Импорт библиотек

In [1]:
!pip install OpenAI
!pip install dotenv
!pip install langfuse


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import json
import pandas as pd
import sys, os
from dotenv import load_dotenv
from datetime import datetime
from openai import OpenAI

In [3]:
PROJECT_ROOT = os.path.abspath("..")  # или путь к корню проекта
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [4]:
load_dotenv()

True

Подключение baseline модели

In [5]:
from ml.models.baseline import baseline_model

Клиент для Mistral (LLM-as-a-Judge)

In [6]:
client_mistral = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=os.getenv("MISTRAL_API_KEY")
)

Few-shot для Mistral

In [7]:
LLM_JUDGE_PROMPT = """
Ты — эксперт-диетолог, проверяющий корректность плана питания, составленного другой моделью.

# Описание критериев:
- **goal_match** — соответствует ли план цели пользователя (например, похудение → умеренный дефицит калорий).
- **meal_type_correctness** — логичны ли блюда (завтрак — лёгкий, ужин — не слишком калорийный).
- **calorie_match** — соответствует ли общая калорийность заявленной пользователем (допуск ±10%).
- **preferences_respected** — соблюдены ли пищевые предпочтения.
- **allergies_respected** — отсутствуют запрещённые ингредиенты.
- **diversity** — блюда разнообразны, не повторяются."""

Схема для JSON-файла

In [8]:
MISTRAL_SCHEMA = {
    "type": "object",
    "properties": {
        "goal_match": {"type": "boolean"},
        "meal_type_correctness": {"type": "boolean"},
        "calorie_match": {"type": "boolean"},
        "preferences_respected": {"type": "boolean"},
        "allergies_respected": {"type": "boolean"},
        "diversity": {"type": "boolean"},
        "comments": {"type": "string"}
    },
    "required": [
        "goal_match", "meal_type_correctness", "calorie_match",
        "preferences_respected", "allergies_respected", "diversity", "comments"
    ]
}

Метод для Mistral (LLM is a judge)

In [9]:
def evaluate_with_mistral(user_data: dict, plan_json: dict):
    """Оценивает план с помощью Mistral."""
    prompt = (
        LLM_JUDGE_PROMPT
        + "\n\n# Данные пользователя:\n"
        + json.dumps(user_data, ensure_ascii=False, indent=2)
        + "\n\n# Сгенерированный план:\n"
        + json.dumps(plan_json, ensure_ascii=False, indent=2)
    )

    response = client_mistral.chat.completions.create(
        model="mistral-medium",
        messages=[
            {"role": "system", "content": "Ты помощник-оценщик, отвечай строго JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=800,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "MealPlanEvaluation",
                "schema": MISTRAL_SCHEMA
            }
        }
    )

    content = response.choices[0].message.content
    if isinstance(content, str):
        return json.loads(content)
    return content


def calculate_overall_score(evaluation: dict) -> float:
    """Вычисляет долю True среди булевых метрик."""
    bools = [v for v in evaluation.values() if isinstance(v, bool)]
    return round(sum(bools) / len(bools), 2) if bools else 0.0


5 тестовых запросов

In [10]:
TEST_USERS = [
    {"goal": "Похудение", "calories": 1800, "budget": 1500, "preferences": "вегетарианская диета", "allergies": "орехи"},
    {"goal": "Набор массы", "calories": 2800, "budget": 3000, "preferences": "мясо, рыба", "allergies": ""},
    {"goal": "Поддержание веса", "calories": 2200, "budget": 2500, "preferences": "средиземноморская диета", "allergies": "лактоза"},
    {"goal": "Похудение", "calories": 1600, "budget": 2000, "preferences": "веганская диета", "allergies": ""},
    {"goal": "Набор массы", "calories": 3000, "budget": 4000, "preferences": "смешанная диета", "allergies": "глютен"}
]

Проверка GPT-4o-mini

In [11]:
import time

results = []

for i, user in enumerate(TEST_USERS, start=1):
    print(f"\nТест #{i}: {user['goal']}")

    start = time.time()
    plan = baseline_model.generate(user_data=user, user_id=f"test_{i}")
    latency = round(time.time() - start, 2)

    eval_result = evaluate_with_mistral(user, plan)
    overall = calculate_overall_score(eval_result)

    results.append({
        "test_case": i,
        "goal": user["goal"],
        "latency_sec": latency,
        "overall": overall,
        "comments": eval_result.get("comments", "")
    })

    # --- Задержка перед следующим тестом ---
    if i < len(TEST_USERS):
        print("Ожидание 10 секунд перед следующим тестом...\n")
        for s in range(10, 0, -1):
            print(f"  {s} сек", end="\r")
            time.sleep(1)



Тест #1: Похудение
Недельный план успешно сгенерирован
Ожидание 10 секунд перед следующим тестом...

  1 секк
Тест #2: Набор массы
Недельный план успешно сгенерирован
Ожидание 10 секунд перед следующим тестом...

  1 секк
Тест #3: Поддержание веса
Недельный план успешно сгенерирован
Ожидание 10 секунд перед следующим тестом...

  1 секк
Тест #4: Похудение
Недельный план успешно сгенерирован
Ожидание 10 секунд перед следующим тестом...

  1 секк
Тест #5: Набор массы
Недельный план успешно сгенерирован


Вывод результатов

In [12]:
df = pd.DataFrame(results)
df = df.sort_values("overall", ascending=False)
df

,test_case,goal,latency_sec,overall,comments
0,1,Похудение,39.53,1.0,План питания соответствует всем критериям. Он ...
1,2,Набор массы,45.39,1.0,План питания соответствует всем критериям. Он ...
2,3,Поддержание веса,46.75,1.0,План питания соответствует всем критериям и пр...
3,4,Похудение,40.65,1.0,План питания соответствует всем критериям. Он ...
4,5,Набор массы,13.14,1.0,"План питания соответствует цели набора массы, ..."


Сохранение результатов

In [14]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
output_path = f"evaluation_results_{timestamp}.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Результаты сохранены в {output_path}")


Результаты сохранены в evaluation_results_20251107_1742.csv
